In [2]:
import pandas as pd
from datasets import load_dataset, Dataset
from typing import Any

In [3]:
train_ds = load_dataset(
    "parquet",
    data_files={"train": "../data/train.parquet"}
)["train"]

val_ds = load_dataset(
    "parquet",
    data_files={"validation": "../data/validation.parquet"}
)["validation"]

test_ds = load_dataset(
    "json",
    data_files={"test": "../data/test.json"}
)["test"]

In [4]:
_UNSET = object
def count_dataset_feature(dataset: Dataset, feature_name: str, value: Any = _UNSET):
    
    if value == _UNSET:
        unique_vals = set(dataset[feature_name])
    else:
        unique_vals = set([value])
    
    
    count_dict = {}
    for val in unique_vals:
        count = dataset[feature_name].count(val)
        
        count_dict[val] = (count, count / dataset.num_rows)

    return count_dict

In [5]:
print(f"Number of rows train: {train_ds.num_rows}")
print(f"Number of rows val: {val_ds.num_rows}")
print(val_ds.features)

Number of rows train: 15343
Number of rows val: 3011
{'question': Value('string'), 'context': Value('string'), 'lang': Value('string'), 'answerable': Value('bool'), 'answer_start': Value('int64'), 'answer': Value('string'), 'answer_inlang': Value('string')}


In [6]:
print("Count numbers for 'answerable' train")
answerable_count_train = count_dataset_feature(train_ds, "answerable")
print(answerable_count_train)
print("Count numbers for 'answerable' val")
answerable_count_val = count_dataset_feature(val_ds, "answerable")
print(answerable_count_val)

print("\nCount numbers for 'answer_inlang' train")
answer_inlang_count_train = count_dataset_feature(train_ds, "answer_inlang", None)
print(f"None counts train: {answer_inlang_count_train[None]}")
print("Count numbers for 'answer_inlang' val")
answer_inlang_count_val = count_dataset_feature(val_ds, "answer_inlang", None)
print(f"None counts val: {answer_inlang_count_val[None]}")

# maybe do standard deviation and average for answer_start?

print("\nCount numbers for 'lang' train")
lang_count_train = count_dataset_feature(train_ds, "lang")
print(lang_count_train)
print("Count numbers for 'lang' val")
lang_count_val = count_dataset_feature(val_ds, "lang")
print(lang_count_val)


Count numbers for 'answerable' train
{False: (1381, 0.09000847291924656), True: (13962, 0.9099915270807535)}
Count numbers for 'answerable' val
{False: (698, 0.23181667220192628), True: (2313, 0.7681833277980737)}

Count numbers for 'answer_inlang' train
None counts train: (15093, 0.9837059245258424)
Count numbers for 'answer_inlang' val
None counts val: (2511, 0.8339422118897376)

Count numbers for 'lang' train
{'te': (1355, 0.08831388906993418), 'ja': (2301, 0.14997067066414652), 'ko': (2422, 0.1578570031936388), 'bn': (2598, 0.16932803232744575), 'fi': (2126, 0.1385648178322362), 'ru': (1983, 0.12924460666101806), 'ar': (2558, 0.16672098025158053)}
Count numbers for 'lang' val
{'te': (384, 0.1275323812686815), 'ja': (456, 0.15144470275655927), 'ko': (356, 0.1182331451345068), 'bn': (476, 0.15808701428096977), 'fi': (528, 0.17535702424443705), 'ru': (396, 0.1315177681833278), 'ar': (415, 0.13782796413151777)}


In [7]:
train_df = pd.read_csv("../data/train_translated.csv")
train_df.rename(columns={train_df.columns[0]: "real_dataset_index"}, inplace=True)
val_df = pd.read_csv("../data/validation_translated.csv")
val_df.rename(columns={val_df.columns[0]: "real_dataset_index"}, inplace=True)

In [8]:
print(train_df.columns)

when_sentences = train_df[train_df['question_translated'].str.contains("what", case=False, na=False)][['context', 'question_translated', 'answer', 'answerable']]
print("Rows where 'question_translated' contains the word 'what':")
print(when_sentences)

when_sentence_count = len(when_sentences)
print(f"The word 'what' appears in {when_sentence_count}")

Index(['real_dataset_index', 'question', 'context', 'lang', 'answerable',
       'answer_start', 'answer', 'answer_inlang', 'question_stripped',
       'question_wordcount', 'context_stripped', 'context_wordcount',
       'question_translated'],
      dtype='object')
Rows where 'question_translated' contains the word 'what':
                                                context  \
3     The British Broadcasting Corporation (BBC) is ...   
5     Andromeda is one of the 48 constellations list...   
6     The Smith–Putnam wind turbine was the world's ...   
7     Louis, Dauphin of France (4 September 1729 – 2...   
10    As a member of a reserve battalion during Worl...   
...                                                 ...   
6321  Vaccines against anthrax for use in livestock ...   
6323  Austin is the capital of the US state of Texas...   
6326  The femur is the longest and, by most measures...   
6331  The first earth tracks were created by humans ...   
6333  Munich (; ; ) is t

In [9]:
split_words = train_df['question_translated'].str.split()

In [10]:
starting_words = split_words.str[0]

starting_word_counts = starting_words.value_counts()

print(f"Number of unique starting words: {starting_word_counts.count()}")

filter_count = 10
filtered_starting_word_counts = starting_word_counts[starting_word_counts > filter_count]
print(f"Number of unique starting words with more than {filter_count}: {filtered_starting_word_counts.count()}")


print(f"Counts of all starting words with more than {filter_count}:")
print(filtered_starting_word_counts)

Number of unique starting words: 80
Number of unique starting words with more than 10: 22
Counts of all starting words with more than 10:
question_translated
What         1596
When         1151
Who           962
How           793
Where         394
What's        379
Is            200
In            143
Which         127
Who's          98
Where's        96
Does           62
Can            44
Why            35
Are            31
Has            27
Did            24
Do             23
As             18
Will           15
Was            12
According      12
Name: count, dtype: int64


In [11]:
starting_words = split_words.str[0]  # Extract the first word

# Count occurrences of each first word where 'answerable' is True
starting_word_counts_answerable = starting_words[train_df['answerable'] == False].value_counts()

print(f"Number of unique starting words where 'answerable' is False: {starting_word_counts_answerable.count()}")

filter_count = 0  # Set the threshold for filtering
filtered_starting_word_counts_answerable = starting_word_counts_answerable[starting_word_counts_answerable > filter_count]

print(f"Number of unique starting words with more than {filter_count} (where 'answerable' is False): {filtered_starting_word_counts_answerable.count()}")

print(f"Counts of all starting words with more than {filter_count} (where 'answerable' is False):")
print(filtered_starting_word_counts_answerable)

Number of unique starting words where 'answerable' is False: 26
Number of unique starting words with more than 0 (where 'answerable' is False): 26
Counts of all starting words with more than 0 (where 'answerable' is False):
question_translated
Is           139
Does          37
Can           37
What          28
Are           22
Do            17
Has           16
Did           15
Who            9
Was            8
How            7
When           4
In             4
Could          3
Which          3
At             2
Were           2
Have           2
From           1
Will           1
Major,         1
Jack           1
What's         1
Who's          1
According      1
As             1
Name: count, dtype: int64


In [12]:
second_words = split_words.str[1]

second_word_counts = second_words.value_counts()  

print(f"Number of unique second words: {second_word_counts.count()}")

filter_count = 10  
filtered_second_word_counts = second_word_counts[second_word_counts > filter_count]

print(f"Number of unique second words with more than {filter_count}: {filtered_second_word_counts.count()}")

print(f"Counts of all second words with more than {filter_count}:")
print(filtered_second_word_counts)

# Print the occurrences of the first word before each second word
for word in filtered_second_word_counts.index:
    first_words_before = split_words[second_words == word].str[0].value_counts()
    print(f"\nOccurrences of first words before the second word '{word}':")
    print(first_words_before)

Number of unique second words: 423
Number of unique second words with more than 10: 34
Counts of all second words with more than 10:
question_translated
is            1600
was           1131
the            666
did            598
many           578
's             164
are            148
what            95
country         87
invented        72
long            63
there           57
which           52
does            48
do              43
year            42
a               35
much            27
were            25
it              22
old             22
founded         21
can             19
of              19
built           15
kind            15
you             13
will            13
big             13
wrote           12
language        12
discovered      12
to              12
far             11
Name: count, dtype: int64

Occurrences of first words before the second word 'is':
question_translated
What             947
Who              369
Where            196
When              43
Which         

In [ ]:
from collections import Counter

# Combine all words in 'context_translated' and 'question_translated' in lowercase
context_words = " ".join(train_df['context'].dropna().str.lower()).split()
question_words = " ".join(train_df['question_translated'].dropna().str.lower()).split()

# Count the occurrences of each word
count_context_words = Counter(context_words)
count_question_words = Counter(question_words)
combined_count = count_context_words + count_question_words

most_common_counts = combined_count.most_common(20)

most_common_words = [word for word, _ in most_common_counts]

max_occurances_least_common = 5
least_common_words = [word for word, count in combined_count.items() if count <= max_occurances_least_common]

most_common_words = list(set(most_common_words))
print(most_common_words)

print(len(least_common_words))


['a', 'are', 'with', 'and', 'it', 'of', 'for', 'on', 'an', 'is', 'first', 'the', 'his', 'that', 'to', 'from', 'in', 'as', 'by', 'was']
64599


Idea is to look at first word of the question to mark what type of question it is;
Is it asking for a date, of what happened, who, where

In [12]:
# Filter rows where 'question_translated' starts with "when" (case-insensitive)
when_questions = train_df[train_df['question_translated'].str.lower().str.startswith("when", na=False)]

# Print the context and question for these rows
print("Contexts and questions where the question starts with 'when':")
row = when_questions[['context', 'question_translated', 'answerable']].iloc[1]
question = row["question_translated"]
context = row["context"]
answerable = row["answerable"]
print(question)
print(context)
print(answerable)

Contexts and questions where the question starts with 'when':
When did the one-day celebration end?
Following seizures of German territories in 1914, the League of Nations granted Japan mandates over some former German possessions in the Western Pacific after World War I. With the Japanese expansion into Manchuria in the early 1930s, Japan adopted a policy of setting up and/or supporting puppet states in conquered regions. In this less obviously imperialist form Japan controlled many of the states of what it referred to as the Greater East Asia Co-Prosperity Sphere, a concept which gradually formed under Japanese influence from 1930 to 1945. Colonial control over the far-flung territories from Tokyo ended after the Allies defeated Japan in 1945: the extent of Japanese governance reverted to the four home islands, the Nanpō Islands, and the Ryukyu Islands.
True


In [ ]:
def rule_based_classifier(context: list, question: list) -> bool:
    
    # match case statement for starting words
    first_token = question[0].lower()
    second_token = question[1].lower()
    question_stripped = [word for word in question if word not in most_common_words]
    match first_token: # matches first token
        case "when":
            
            pass
        case "where":
            # look for places or references to places
            # also followed by is and was
            pass
        case "who":
            # names or references to people (e.g. president of america)
            # also followed by is and was
            pass
        case "what":
            # needs to look at 
            # is usually followed by is or was
            pass
        case "how":
            if second_token == "many":
                # TODO: look for number
                pass
            pass
        case _:
            # base case
            pass
    # go through the context
    # match context with starting words
    pass